# Memory demo: Calculator agent

Same agent, same questions, run twice:

1. **Without memory**: every call starts from a blank slate.
2. **With memory**: a checkpointer saves the conversation, so "that" and "it" make sense.

The questions are deliberately chained ("multiply *that* by 5"), so they only work if the agent remembers the previous answer.

## Setup

This notebook uses **Azure AI Foundry** (OpenAI-compatible `/openai/v1` endpoint). Install the integration once with `pip install langchain-openai python-dotenv`, then add `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY` and `AZURE_OPENAI_DEPLOYMENT` to your `.env` file (see the next cell).

In [ ]:
%%capture
%pip install --upgrade langgraph langchain langchain_openai langchain-community
%pip install --upgrade python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Azure AI Foundry settings - put these in your .env file:
#   AZURE_OPENAI_ENDPOINT=https://<your-resource>.services.ai.azure.com/openai/v1
#   AZURE_OPENAI_API_KEY=<your key>
#   AZURE_OPENAI_DEPLOYMENT=<your deployment name, e.g. gpt-4o>
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# The client wants the base URL only (".../openai/v1"). If the .env value was copied with a
# trailing "/responses" or "/chat/completions", strip it, otherwise the request 404s.
AZURE_OPENAI_BASE_URL = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
for suffix in ("/responses", "/chat/completions"):
    AZURE_OPENAI_BASE_URL = AZURE_OPENAI_BASE_URL.removesuffix(suffix)
print("base_url:", AZURE_OPENAI_BASE_URL)

## Calculator tools

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Add two numbers: a + b."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a: a - b."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers: a * b."""
    return a * b

@tool
def divide(a: float, b: float) -> str | float:
    """Divide a by b: a / b."""
    if b == 0:
        return "Error: cannot divide by zero"
    return a / b

tools = [add, subtract, multiply, divide]

In [ ]:
SYSTEM_PROMPT = """You are a calculator assistant.

Rules:
- ALWAYS use the tools (add, subtract, multiply, divide) for arithmetic. Never calculate in your head.
- Call ONE tool at a time.
- If the user says 'that', 'it' or 'the result', use the result of the previous calculation in this conversation.
- If you do not know which number the user means, ask them. Do not guess.
- After the tool returns, reply with a short sentence containing the answer.
"""

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# Azure AI Foundry's /openai/v1 endpoint speaks the standard OpenAI API,
# so we use ChatOpenAI (not AzureChatOpenAI) and pass the deployment name as `model`.
# temperature=0 keeps tool calling stable for a demo
# (drop it if your deployment is an o-series / gpt-5 model that rejects temperature)
chat = ChatOpenAI(
    base_url=AZURE_OPENAI_BASE_URL,
    api_key=AZURE_OPENAI_API_KEY,
    model=AZURE_OPENAI_DEPLOYMENT,  # the deployment name
    temperature=0,
)

## Helper

`ask()` sends one question and prints the answer. It also prints **how many messages the model received**. This number shows the difference between the two agents.

In [ ]:
def ask(agent, question, thread_id=None):
    """Send one question. Pass thread_id only for an agent that has memory."""
    config = {"configurable": {"thread_id": thread_id}} if thread_id else None

    print(f"\n>>> {question}   (thread_id={thread_id})")
    last = None
    for step in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        config,
        stream_mode="values",
    ):
        last = step

    # tool calls made in THIS turn only (everything after the last human message)
    msgs = last["messages"]
    turn = msgs[max(i for i, m in enumerate(msgs) if m.type == "human"):]
    for m in turn:
        for call in getattr(m, "tool_calls", []) or []:
            print(f"    tool call: {call['name']}({call['args']})")

    print(f"    messages in the agent's state: {len(last['messages'])}")
    print(f"    ANSWER: {last['messages'][-1].content}")

---
# Part 1: Without memory

No `checkpointer`. Every `ask()` starts with only the single question we send.

In [ ]:
agent_no_memory = create_agent(
    model=chat,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

In [ ]:
ask(agent_no_memory, "What is 12 plus 8?")

In [ ]:
ask(agent_no_memory, "What is 12 plus 8 and multiply that by 5")

### What happened without memory

- Question 1 works: `add(12, 8)` gives 20.
- Question 2 fails. The agent never saw question 1, so it does not know what "that" is. It asks which number you mean, or guesses.

Each call was a **brand new conversation**: the message count never grows past one question, its tool call and the answer.

---
# Part 2: With memory

The only code changes:
1. `checkpointer=InMemorySaver()` in `create_agent(...)` turns saving on.
2. `thread_id` in each call says which saved conversation to continue.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent_with_memory = create_agent(
    model=chat,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

In [ ]:
ask(agent_with_memory, "What is 12 plus 8?", thread_id="calc-1")

In [ ]:
ask(agent_with_memory, "Multiply that by 5", thread_id="calc-1")

In [ ]:
ask(agent_with_memory, "Now divide it by 4", thread_id="calc-1")

In [ ]:
ask(agent_with_memory, "Subtract 5 from the result", thread_id="calc-1")

### What happened with memory

- "that" resolved to 20, so `multiply(20, 5)` gives 100.
- "it" resolved to 100, so `divide(100, 4)` gives 25.
- "the result" resolved to 25, so `subtract(25, 5)` gives 20.
- The message count **keeps growing**, because each call loads the saved history, adds the new question and saves it again.

## Look inside the memory

In [ ]:
state = agent_with_memory.get_state({"configurable": {"thread_id": "calc-1"}})

for m in state.values["messages"]:
    if m.type == "ai" and m.tool_calls:
        c = m.tool_calls[0]
        print(f"AI    -> tool call {c['name']}({c['args']})")
    else:
        print(f"{m.type.upper():5} -> {str(m.content)[:80]}")

## Different `thread_id` = different conversation

In [ ]:
ask(agent_with_memory, "Multiply that by 5", thread_id="calc-2")

`calc-2` is a new thread with no history, so the agent is as lost as the no-memory agent. Memory is stored **per thread**: one thread per user or chat session.

---
## Summary

| | Without memory | With memory |
|---|---|---|
| `create_agent(...)` | no `checkpointer` | `checkpointer=InMemorySaver()` |
| `stream/invoke` config | none | `{"configurable": {"thread_id": "..."}}` |
| "Multiply that by 5" | doesn't know what "that" is | uses the previous answer |
| Messages sent to the model | only the current question | the whole thread so far |

`InMemorySaver` lives in RAM, so restarting the kernel erases it. For storage that survives restarts, use `SqliteSaver` or `PostgresSaver`.